# Movie Recommendation System
### Content-Based Filtering — TMDB API
---

In [18]:
import numpy as np
import pandas as pd
import requests
import pickle
import time
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from nltk.stem.porter import PorterStemmer
import nltk
nltk.download('punkt', quiet=True)

True

In [19]:
API_KEY  = 'ebef1aa0b639138c3040e6929ea9f1eb'
BASE_URL = 'https://api.themoviedb.org/3'
IMG_BASE = 'https://image.tmdb.org/t/p/w500'

## Step 1 — Fetch Movie List from TMDB

In [20]:
def get_data_list(media_type, endpoint, pages=200):
    items = []
    for page in range(1, pages + 1):
        r = requests.get(
            f'{BASE_URL}/{media_type}/{endpoint}',
            params={'api_key': API_KEY, 'language': 'en-US', 'page': page}
        )
        results = r.json().get('results', [])
        for item in results:
            item['media_type'] = media_type
        items.extend(results)
        time.sleep(0.05)
    return items

movies_popular     = get_data_list('movie', 'popular',      pages=150)
movies_top_rated   = get_data_list('movie', 'top_rated',    pages=150)
movies_now_playing = get_data_list('movie', 'now_playing',  pages=20)   # latest theatrical releases

tv_popular         = get_data_list('tv', 'popular',         pages=150)
tv_top_rated       = get_data_list('tv', 'top_rated',       pages=100)
tv_airing_today    = get_data_list('tv', 'airing_today',    pages=20)   # latest tv releases

all_items_list = movies_popular + movies_top_rated + movies_now_playing + tv_popular + tv_top_rated + tv_airing_today
# Deduplicate by id and media_type (since TMDB ids are unique per media type, we use a tuple key)
all_items = {(m['id'], m['media_type']): m for m in all_items_list}
print(f'Total unique items: {len(all_items)}')

Total unique items: 9300


## Step 2 — Fetch Details, Credits & Keywords

In [21]:
def fetch_item_data(item_id, media_type):
    params = {'api_key': API_KEY, 'language': 'en-US'}
    detail   = requests.get(f'{BASE_URL}/{media_type}/{item_id}',          params=params).json()
    credits  = requests.get(f'{BASE_URL}/{media_type}/{item_id}/credits',  params=params).json()
    keywords = requests.get(f'{BASE_URL}/{media_type}/{item_id}/keywords', params=params).json()
    return detail, credits, keywords

records = []

for i, (key, m) in enumerate(all_items.items()):
    item_id, media_type = key
    try:
        detail, credits, kw = fetch_item_data(item_id, media_type)
        genres   = [g['name'] for g in detail.get('genres', [])]

        # Keywords format is different for TV and Movies
        kw_key   = 'results' if media_type == 'tv' else 'keywords'
        keywords = [k['name'] for k in kw.get(kw_key, [])]

        cast     = [c['name'] for c in credits.get('cast', [])[:3]]

        # Crew logic (Director or Creator)
        if media_type == 'movie':
            crew_members = [c['name'] for c in credits.get('crew', []) if c.get('job') == 'Director'][:1]
        else:
            crew_members = [c['name'] for c in detail.get('created_by', [])][:1]
            if not crew_members:
                crew_members = [c['name'] for c in credits.get('crew', []) if c.get('job') in ['Director', 'Executive Producer']][:1]

        title = detail.get('title', '') if media_type == 'movie' else detail.get('name', '')

        records.append({
            'movie_id':     item_id,
            'media_type':   media_type,
            'title':        title,
            'overview':     detail.get('overview', ''),
            'poster_path':  detail.get('poster_path', ''),
            'vote_average': detail.get('vote_average', 0),
            'genres':       genres,
            'keywords':     keywords,
            'cast':         cast,
            'crew':         crew_members,
        })
        time.sleep(0.05)
        if (i + 1) % 100 == 0:
            print(f'{i+1}/{len(all_items)} done...')
    except Exception as e:
        print(f'Skipped {item_id} ({media_type}): {e}')

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
df.head(3)

100/9300 done...
200/9300 done...
300/9300 done...
400/9300 done...
500/9300 done...
600/9300 done...
700/9300 done...
800/9300 done...
900/9300 done...
Skipped 676 (movie): HTTPSConnectionPool(host='api.themoviedb.org', port=443): Read timed out. (read timeout=None)
Skipped 1294203 (movie): HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/1294203?api_key=ebef1aa0b639138c3040e6929ea9f1eb&language=en-US (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001B2D6788E10>: Failed to resolve 'api.themoviedb.org' ([Errno 11001] getaddrinfo failed)"))
Skipped 1401586 (movie): HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/1401586?api_key=ebef1aa0b639138c3040e6929ea9f1eb&language=en-US (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001B2D6789090>: Failed to resolve 'api.themoviedb.org' ([Errno 11001] getaddrinfo failed)"))
Skippe

,movie_id,media_type,title,overview,poster_path,vote_average,genres,keywords,cast,crew
0,1523145,movie,Your Heart Will Be Broken,High school student Polina is saved from bully...,/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg,5.556,"[Romance, Drama]",[],"[Veronika Zhuravleva, Daniel Vegas, Alya Mayer]",[Mikhail Vaynberg]
1,875828,movie,Peaky Blinders: The Immortal Man,After his estranged son gets embroiled in a Na...,/gRMalasZEzsZi4w2VFuYusfSfqf.jpg,7.363,"[Crime, Drama]","[england, gangster, world war ii, period drama...","[Cillian Murphy, Barry Keoghan, Rebecca Ferguson]",[Tom Harper]
2,83533,movie,Avatar: Fire and Ash,In the wake of the devastating war against the...,/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg,7.260,"[Science Fiction, Adventure, Fantasy]","[witch, clone, space war, tribe, sequel, alien...","[Sam Worthington, Zoe Saldaña, Sigourney Weaver]",[James Cameron]


## Step 3 — Clean Data

In [ ]:
df = df[df['overview'].str.strip() != ''].dropna(subset=['overview']).reset_index(drop=True)
print(f'After cleaning: {df.shape}')
df.isnull().sum()

## Step 4 — Build Tags

In [ ]:
for col in ['genres', 'keywords', 'cast', 'crew']:
    df[col] = df[col].apply(lambda lst: [x.replace(' ', '') for x in lst])

# Feature weighting: repeat high-signal fields so TF-IDF scores them higher
# genres × 3, director × 3, cast × 2, keywords × 2, overview × 1
df['tags'] = (
    df['overview'].apply(str.split) +
    df['genres']   * 3 +
    df['crew']     * 3 +
    df['cast']     * 2 +
    df['keywords'] * 2
)
df['tags'] = df['tags'].apply(lambda x: ' '.join(x).lower())
df[['title', 'tags']].head(3)

## Step 5 — Stemming

In [ ]:
ps = PorterStemmer()
df['tags'] = df['tags'].apply(lambda text: ' '.join(ps.stem(w) for w in text.split()))
print(df.loc[0, 'tags'][:300])

## Step 6 — Vectorize & Cosine Similarity

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors    = cv.fit_transform(df['tags']).toarray()
similarity = cosine_similarity(vectors)
print(f'Vectors:    {vectors.shape}')
print(f'Similarity: {similarity.shape}')

## Step 7 — Test Recommendation

In [ ]:
def recommend(title, n=5):
    match = df[df['title'].str.lower() == title.lower()]
    if match.empty:
        print(f"'{title}' not found.")
        return
    idx   = match.index[0]
    top_n = sorted(enumerate(similarity[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    print(f'Recommendations for "{title}":')
    for rank, (i, score) in enumerate(top_n, 1):
        print(f'  {rank}. {df.iloc[i].title}  ({score:.3f})')

recommend('Avatar')

## Step 8 — Save Model

In [ ]:
movies_export = df[['movie_id', 'media_type', 'title', 'overview', 'poster_path', 'vote_average']].copy()
pickle.dump(movies_export.to_dict(), open('movies_dict.pkl', 'wb'))
pickle.dump(similarity,              open('similarity.pkl',  'wb'))
print(f'Saved movies_dict.pkl  ({len(movies_export)} items)')
print('Saved similarity.pkl')